In [8]:
import pandas as pd
import tomllib
from pathlib import Path

# Read Google Sheets settings
with open(".streamlit/secrets.toml", "rb") as f:
    secrets = tomllib.load(f)

# Get non-transfer spreadsheet information
sheet = secrets["connections"]["non_transfers"]

spreadsheet_id = sheet["spreadsheet"].split("/d/")[1].split("/")[0]
gid = sheet["worksheet"]

# Create Google Sheets CSV URL
csv_url = (
    f"https://docs.google.com/spreadsheets/d/{spreadsheet_id}"
    f"/export?format=csv&gid={gid}"
)

# Load non-transfer data into pandas
non_transfers = pd.read_csv(csv_url)

# Check dataset size
non_transfers.shape

(8037, 72)

In [11]:
# Columns needed for Nearest Neighbors matching
matching_columns = [
    "id",
    "name",
    "position",
    "season",
    "team",
    "power_conf",
    "prev_season",
    "prev_team",
    "prev_power_conf",
    "transferred",
    "passing_ATT",
    "passing_YDS",
    "passing_YPA",
    "passing_TD",
    "rushing_CAR",
    "rushing_YDS",
    "rushing_YPC",
    "rushing_TD",
    "receiving_REC",
    "receiving_YDS",
    "receiving_YPR",
    "receiving_TD",
    "adjPPA"
]

non_transfer_matching = non_transfers[matching_columns].copy()

In [12]:
# Match non-transfer players with their previous-season statistics
non_transfer_pairs = non_transfers.merge(
    non_transfers,
    left_on=["id", "prev_season", "prev_team"],
    right_on=["id", "season", "team"],
    suffixes=("", "_before"),
    how="inner"
)

In [13]:
# Match non-transfer players with their previous-season statistics
non_transfer_pairs = non_transfers.merge(
    non_transfers,
    left_on=["id", "prev_season", "prev_team"],
    right_on=["id", "season", "team"],
    suffixes=("", "_before"),
    how="inner"
)

In [ ]:
# Check missing values in the performance metrics
stat_columns = [
    "passing_ATT", "passing_YDS",
    "rushing_CAR", "rushing_YDS",
    "receiving_REC", "receiving_YDS"
]

non_transfer_pairs[
    stat_columns + [f"{col}_before" for col in stat_columns]
].isna().sum()

passing_ATT             2872
passing_YDS             2872
rushing_CAR             1454
rushing_YDS             1454
receiving_REC            654
receiving_YDS            654
passing_ATT_before      2914
passing_YDS_before      2914
rushing_CAR_before      1526
rushing_YDS_before      1526
receiving_REC_before     806
receiving_YDS_before     806
dtype: int64

In [15]:
# Required statistics for each position
required_stats = {
    "QB": ["passing_ATT", "passing_YDS"],
    "RB": ["rushing_CAR", "rushing_YDS"],
    "WR": ["receiving_REC", "receiving_YDS"],
    "TE": ["receiving_REC", "receiving_YDS"]
}

clean_groups = []

for position, stats in required_stats.items():
    group = non_transfer_pairs[
        non_transfer_pairs["position"] == position
    ].copy()

    # Require statistics from both seasons
    columns = stats + [f"{stat}_before" for stat in stats]

    group = group.dropna(subset=columns)

    clean_groups.append(group)

non_transfer_clean = pd.concat(clean_groups, ignore_index=True)

non_transfer_clean.shape

(3209, 143)

In [17]:
# Check transfer status
non_transfer_clean.head()

,season,id,name,position,team,fbs_conference,power_conf,transferred,transfer_in,prev_season,...,kickReturns_NO_before,kickReturns_TD_before,kickReturns_YDS_before,puntReturns_AVG_before,puntReturns_LONG_before,puntReturns_NO_before,puntReturns_TD_before,puntReturns_YDS_before,plays_before,adjPPA_before
0,2024,4819250,AJ Bianco,QB,Nevada,Mountain West,N,N,N,2023.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,118,0.277724
1,2025,4819250,AJ Bianco,QB,Nevada,Mountain West,N,N,N,2024.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28,0.300712
2,2025,5146355,AJ Hairston,QB,Massachusetts,Mid-American,N,N,N,2024.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,95,0.274866
3,2023,4869321,AJ Padgett,QB,Rice,American Athletic,N,N,N,2022.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,107,0.239097
4,2025,5132812,Aaron Philo,QB,Georgia Tech,ACC,Y,N,N,2024.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,85,0.290283


In [18]:
# Count non-transfer comparisons by position and conference level
non_transfer_clean.groupby(
    ["position", "power_conf"]
).size()

position  power_conf
QB        N             187
          Y             184
RB        N             454
          Y             499
TE        N             291
          Y             365
WR        N             626
          Y             603
dtype: int64

In [19]:
# Player information
info_columns = [
    "id", "name", "position",
    "season", "season_before",
    "team", "power_conf", "power_conf_before"
]

# Performance statistics
stats = [
    "passing_ATT", "passing_YDS", "passing_YPA", "passing_TD",
    "rushing_CAR", "rushing_YDS", "rushing_YPC", "rushing_TD",
    "receiving_REC", "receiving_YDS", "receiving_YPR", "receiving_TD",
    "adjPPA"
]

# Keep before and after statistics
columns = info_columns + stats + [f"{col}_before" for col in stats]

non_transfer_clean = non_transfer_clean[columns].copy()

non_transfer_clean.shape

(3209, 34)

In [20]:
non_transfer_clean.columns.tolist()

['id',
 'name',
 'position',
 'season',
 'season_before',
 'team',
 'power_conf',
 'power_conf_before',
 'passing_ATT',
 'passing_YDS',
 'passing_YPA',
 'passing_TD',
 'rushing_CAR',
 'rushing_YDS',
 'rushing_YPC',
 'rushing_TD',
 'receiving_REC',
 'receiving_YDS',
 'receiving_YPR',
 'receiving_TD',
 'adjPPA',
 'passing_ATT_before',
 'passing_YDS_before',
 'passing_YPA_before',
 'passing_TD_before',
 'rushing_CAR_before',
 'rushing_YDS_before',
 'rushing_YPC_before',
 'rushing_TD_before',
 'receiving_REC_before',
 'receiving_YDS_before',
 'receiving_YPR_before',
 'receiving_TD_before',
 'adjPPA_before']

In [21]:
# Arrange before and after statistics side by side
columns = info_columns.copy()

for stat in stats:
    columns.extend([f"{stat}_before", stat])

non_transfer_clean = non_transfer_clean[columns].copy()

non_transfer_clean.columns.tolist()

['id',
 'name',
 'position',
 'season',
 'season_before',
 'team',
 'power_conf',
 'power_conf_before',
 'passing_ATT_before',
 'passing_ATT',
 'passing_YDS_before',
 'passing_YDS',
 'passing_YPA_before',
 'passing_YPA',
 'passing_TD_before',
 'passing_TD',
 'rushing_CAR_before',
 'rushing_CAR',
 'rushing_YDS_before',
 'rushing_YDS',
 'rushing_YPC_before',
 'rushing_YPC',
 'rushing_TD_before',
 'rushing_TD',
 'receiving_REC_before',
 'receiving_REC',
 'receiving_YDS_before',
 'receiving_YDS',
 'receiving_YPR_before',
 'receiving_YPR',
 'receiving_TD_before',
 'receiving_TD',
 'adjPPA_before',
 'adjPPA']